<a href="https://colab.research.google.com/github/1-THEBEST/DEVSOC-Vertical-Assignment/blob/AI-ML/Copy_of_Lin_Reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Regression

In [ ]:
# Import the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



## Data Preprocessing

### **Exploring the dataset**

Let's start with loading the training data from the csv into a pandas dataframe



Load the datasets from GitHub. Train dataset has already been loaded for you in df below. To get test dataset use the commented code.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/train_processed_splitted.csv')

Let's see what the first 5 rows of this dataset looks like

In [ ]:
df

,LotArea,TotalBsmtSF,GrLivArea,GarageArea,PoolArea,OverallCond,Utilities,SalePrice
0,11553,1051,1159,336,0,5,AllPub,158000
1,8400,1052,1052,288,0,5,AllPub,138500
2,8960,1008,1028,360,0,6,AllPub,115000
3,11100,0,930,308,0,7,AllPub,84900
4,15593,1304,2287,667,0,4,AllPub,225000
...,...,...,...,...,...,...,...,...
1309,9020,1127,1165,490,0,7,AllPub,174900
1310,10793,780,1620,462,0,5,AllPub,152000
1311,8885,864,902,484,0,5,AllPub,131000
1312,11275,710,2978,564,0,7,AllPub,242000


What are all the features present? What is the range for each of the features along with their mean?

In [ ]:
#print((df.head()))
for column, values in df.items():
  if type(values[0]) != str:
    sum = 0
    i = 1
    print(f"column: {column}")
    max = df[column].max()
    min = df[column].min()
    print(f"max: {max}\nmin: {min}")
    for value in values:
      sum += value
      i+=1
    average = sum/i
    Range = max - min
    print(f"range: {Range} , [{min} - {max}]")
    print(f"average: {average}")
    print("\n")

column: LotArea
max: 215245
min: 1300
range: 213945 , [1300 - 215245]
average: 10614.026615969582


column: TotalBsmtSF
max: 6110
min: 0
range: 6110 , [0 - 6110]
average: 1057.506463878327


column: GrLivArea
max: 5642
min: 334
range: 5308 , [334 - 5642]
average: 1511.7498098859317


column: GarageArea
max: 1418
min: 0
range: 1418 , [0 - 1418]
average: 473.12015209125474


column: PoolArea
max: 738
min: 0
range: 738 , [0 - 738]
average: 2.64106463878327


column: OverallCond
max: 9
min: 1
range: 8 , [1 - 9]
average: 5.577946768060836


column: SalePrice
max: 755000
min: 34900
range: 720100 , [34900 - 755000]
average: 180658.0174904943




### **Feature Scaling and One-Hot Encoding**

You must have noticed that some features `(such as Utilities)` are not continuous values.
  
These features contain values indicating different categories and must somehow be converted to numbers so that the computer can understand it. `(Computers only understand numbers and not strings)`
  
These features are called categorical features. We can represent these features as a `One-Hot Representation`
  
  
You must have also noticed that all the other features, each are in a different scale. This can be detremental to the performance of our linear regression model and so we normalize them so that all of them are in the range $[0,1]$

> NOTE: When you are doing feature scaling, store the min/max which you will use to normalize somewhere. This is then to be used at testing time. Try to think why are doing this?

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(handle_unknown='ignore' , sparse_output=False).set_output(transform='pandas')
# Do the one-hot encoding here
#df['Utilities'].unique()
ohetransform = ohe.fit_transform(df[['Utilities']])
ohetransform
df = pd.concat([df, ohetransform], axis = 1).drop(columns=['Utilities'])

In [ ]:
# Do the feature scaling here
Sale_max = df['SalePrice'].max()
Sale_Min = df['SalePrice'].min()
for column, values in df.items():
  print(f"{column}")
  # Convert the column to float type before scaling #gemini suggestion
  df[column] = df[column].astype(float)             #gemini suggestion
  max = df[column].max()
  min = df[column].min()
  for index, value in enumerate(values):
    value = (value - min)/(max - min)
    df.loc[index , column] = value

LotArea
TotalBsmtSF
GrLivArea
GarageArea
PoolArea
OverallCond
SalePrice
Utilities_AllPub
Utilities_NoSeWa


In [ ]:
#checking if feature scaling was correctly implemented
print(df.head())
#print(df['SalePrice'].max())

    LotArea  TotalBsmtSF  GrLivArea  GarageArea  PoolArea  OverallCond  \
0  0.047924     0.172013   0.155426    0.236953       0.0        0.500   
1  0.033186     0.172177   0.135268    0.203103       0.0        0.500   
2  0.035804     0.164975   0.130746    0.253879       0.0        0.625   
3  0.045806     0.000000   0.112283    0.217207       0.0        0.750   
4  0.066807     0.213421   0.367935    0.470381       0.0        0.375   

   SalePrice  Utilities_AllPub  Utilities_NoSeWa  
0   0.170948               1.0               0.0  
1   0.143869               1.0               0.0  
2   0.111235               1.0               0.0  
3   0.069435               1.0               0.0  
4   0.263991               1.0               0.0  


### **Conversion to NumPy**

Ok so now that we have all preprocessed all the data, we need to convert it to numpy for our linear regression model
  
Assume that our dataset has a total of $N$ datapoints. Each datapoint having a total of $D$ features (after one-hot encoding), we want our numpy array to be of shape $(N, D)$

In our task, we have to predict the `SalePrice`. We will need 2 numpy arrays $

*   List item
*   List item

(X, Y)$. These represent the features and targets respectively

In [ ]:
# Convert to numpy array
features = df.to_numpy()
#print(features)

target = np.array(df['SalePrice'])
#print(target)

## Linear Regression formulation
  
We now have our data in the form we need. Let's try to create a linear model to get our initial (Really bad) prediction


Let's say a single datapoint in our dataset consists of 3 features $(x_1, x_2, x_3)$, we can pose it as a linear equation as follows:
$$ y = w_1x_1 + w_2x_2 + w_3x_3 + b $$
Here we have to learn 4 parameters $(w_1, w_2, w_3, b)$
  
  
Now how do we extend this to multiple datapoints?  
  
  
Try to answer the following:
- How many parameters will we have to learn in the cae of our dataset? (Don't forget the bias term)
- Form a linear equation for our dataset. We need just a single matrix equation which correctly represents all the datapoints in our dataset
- Implement the linear equation as an equation using NumPy arrays (Start by randomly initializing the weights from a standard normal distribution)

In [ ]:
datapoints = (features.shape)[1]
print(datapoints)
weights = np.random.randn(1,datapoints)
bias = np.random.randn(1,1)
print(weights)
print(bias)
prediction = features @ weights.T + bias
#print(prediction)
for i in range(0,5):
  print(prediction[i,0])


9
[[ 0.53106208 -0.88939085  1.930356    0.58228117 -1.13162589 -0.03483962
  -0.38605379  0.42009364  1.26887307]]
[[-0.93587284]]
-0.28873021394008647
-0.3448712043576758
-0.30799501133720364
-0.20116599694349224
0.19904522811401404


How well does our model perform? Try comparing our predictions with the actual values

In [ ]:
#print(prediction.shape , target.shape)
print("Comparing first 5 with actual SalePrice:")
for i in range(5):
    print(f"Predicted: {prediction[i,0]}, Actual: {target[i]}")

Comparing first 5 with actual SalePrice:
Predicted: -0.28873021394008647, Actual: 0.1709484793778642
Predicted: -0.3448712043576758, Actual: 0.14386890709623662
Predicted: -0.30799501133720364, Actual: 0.11123455075683933
Predicted: -0.20116599694349224, Actual: 0.06943480072212192
Predicted: 0.19904522811401404, Actual: 0.2639911123455076


### **Learning weights using gradient descent**

So these results are really horrible. We need to somehow update our weights so that it correclty represents our data. How do we do that?

We must do the following:
- We need some numerical indication for our performance, for this we define a Loss Function ( $\mathscr{L}$ )
- Find the gradients of the `Loss` with respect to the `Weights`
- Update the weights in accordance to the gradients: $W = W - \alpha\nabla_W \mathscr{L}$

Lets define the loss function:
- We will use the MSE loss since it is a regression task. (Specify the assumptions we make while doing so as taught in the class).
- Implement this loss as a function. (Use numpy as much as possible)

In [ ]:
def mse_loss_fn(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

print(mse_loss_fn(target , prediction))

0.1596546915263684


Calculate the gradients of the loss with respect to the weights (and biases). First write the equations down on a piece of paper, then proceed to implement it

In [ ]:
def get_gradients(y_true, y_pred, W, b, X):
    """
    Calculates the gradients for the MSE loss function with respect to the weights (and bias)

    Args:
        y_true: The true values of the target variable (SalePrice in our case)
        y_pred: The predicted values of the target variable using our model (W*X + b)

        W: The weights of the model
        b: The bias of the model
        X: The input features

    Returns:
        dW: The gradients of the loss function with respect to the weights
        db: The gradients of the loss function with respect to the bias
    """

Update the weights using the gradients

In [ ]:
def update(weights, bias, gradients_weights, gradients_bias, lr):
    """
    Updates the weights (and bias) using the gradients and the learning rate

    Args:
        weights: The current weights of the model
        bias: The current bias of the model

        gradients_weights: The gradients of the loss function with respect to the weights
        gradients_bias: The gradients of the loss function with respect to the bias

        lr: The learning rate

    Returns:
        weights_new: The updated weights of the model

    """

Put all these together to find the loss value, its gradient and finally updating the weights in a loop. Feel free to play around with different learning rates and epochs
  
> NOTE: The code in comments are just meant to be used as a guide. You will have to do changes based on your code

In [ ]:
NUM_EPOCHS = 1000
LEARNING_RATE = 2e-2

losses = []

for epoch in range(NUM_EPOCHS):
    y_pred = x @ w.T + b
    loss = mse_loss_fn(y, y_pred)
    losses.append(loss)
    dw, db = get_gradients(y, y_pred, w, b, x)
    w, b = update(w, b, dw, db, LEARNING_RATE)


Now use matplotlib to plot the loss graph

In [ ]:
plt.plot(losses)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

### **Testing with test data**

Load and apply all the preprocessing steps used in the training data for the testing data as well. Remember to use the **SAME** min/max values which you used for the training set and not recalculate them from the test set. Also mention why we are doing this.

To load test data from GitHub, use the code below.


In [ ]:
df_test = pd.read_csv('https://raw.githubusercontent.com/cronan03/DevSoc_AI-ML/main/test_processed_splitted.csv')
print(df_test)

# Let's find all the columns that are missing in the test set
missing_cols = set(df.columns) - set(df_test.columns)

# Add these columns to the test set with all zeros
for col in missing_cols:
    df_test[col] = 0

if 'Utilities_AllPub' not in df_test.columns:
    df_test = df_test.join(pd.get_dummies(df_test['Utilities'], dtype = 'int32', prefix = 'Utilities'))
    df_test = df_test.drop('Utilities', axis = 1)



Using the weights learnt above, predict the values in the test dataset. Also answer the following questions:
- Are the predictions good?
- What is the MSE loss for the testset
- Is the MSE loss for testing greater or lower than training
- Why is this the case

In [ ]:
# Scale the features

# Fill NaN values
df_test.fillna(0, inplace=True)

# Scale features


# Check for unexpected NaNs




# Convert to numpy array
x_test = df_test.copy().drop('SalePrice', axis=1).to_numpy() # (N, D)
y_test = df_test.copy()['SalePrice'].to_numpy().reshape(-1, 1) # (N, 1)
print(x_test.shape)


In [ ]:
extra_cols = list(set(df_test.columns) - set(df.columns))
print("Extra columns in df_test:", extra_cols)

missing_cols = list(set(df.columns) - set(df_test.columns))
print("Missing columns in df_test:", missing_cols)

In [ ]:
# Make predictions
y_pred_test = x_test @ w.T + b # (N, 1)
loss_test = mse_loss_fn(y_pred_test, y_test)


# Scale the predictions back to the original scale


In [ ]:
idx = np.random.randint(0, x_test.shape[0], 5)
y_pred_test_sample = y_pred_test_scaled[idx].round().astype(int)
y_true_test_sample = y_test_scaled[idx].round().astype(int)

print('Predicted SalePrice: \t', y_pred_test_sample.squeeze().tolist())
print('Actual SalePrice: \t', y_true_test_sample.squeeze().tolist())
print('\nTest Loss: \t\t', loss_test)